# anatomix + FireANTs: 3D multimodal registration

This tutorial registers a 3D volume pair with **`anatomix-register.py`**, the FireANTs-based registration CLI. It runs end-to-end on a Google Colab **T4 (16 GB)** GPU using a small synthetic pair and T4-safe settings.

> Runtime → Change runtime type → **T4 GPU** before running.

## 1. Install anatomix and the FireANTs backend

In [ ]:
# Run top-to-bottom once. Clones anatomix and installs it WITHOUT reinstalling
# Colab's torch (so the runtime does not need a restart).
import os
if not os.path.isdir("/content/anatomix"):
    !git clone https://github.com/neel-dey/anatomix.git /content/anatomix
%cd /content/anatomix
!pip -q install -e . --no-deps
!pip -q install monai "dynamic-network-architectures==0.4.4" nibabel SimpleITK \
    torchio natsort huggingface_hub scikit-learn scipy scikit-image matplotlib

In [ ]:
# Install the FireANTs backend (gitignored editable clone; protects torch).
!bash anatomix/registration/registration_backend/install_fireants.sh

## 2. Create a small synthetic demo pair

`fixed` is a textured shape; `moving` is a smoothly-deformed, contrast-inverted copy (a cross-modality proxy) with matching masks and segmentations.

In [ ]:
import os
import numpy as np
import nibabel as nib
from scipy.ndimage import gaussian_filter, binary_dilation, map_coordinates

def make_demo_data(outdir="demo", size=128, seed=0):
    """Write a small synthetic fixed/moving pair with masks and segmentations.

    The moving image is a smoothly-deformed, contrast-inverted version of the
    fixed image (a cross-modality proxy), so anatomix's contrast-invariant
    features have something meaningful to align.
    """
    os.makedirs(outdir, exist_ok=True)
    rng = np.random.default_rng(seed)
    lin = np.linspace(-1, 1, size)
    zz, yy, xx = np.meshgrid(lin, lin, lin, indexing="ij")
    r = np.sqrt(xx ** 2 + yy ** 2 + zz ** 2)

    outer = np.sqrt((xx / 0.7) ** 2 + (yy / 0.6) ** 2 + (zz / 0.65) ** 2) < 1
    inner = r < 0.28
    seg = outer.astype(np.int16) + inner.astype(np.int16)  # 0 bg, 1 shell, 2 core

    texture = gaussian_filter(rng.standard_normal((size,) * 3), 3)
    fixed = outer * (0.6 + 0.4 * np.sin(6 * xx) * np.cos(6 * yy)) + 0.3 * texture * outer
    fixed = np.clip(fixed, 0, None).astype(np.float32)

    grid = np.stack(np.meshgrid(*[np.arange(size)] * 3, indexing="ij")).astype(np.float32)
    disp = np.stack([gaussian_filter(rng.standard_normal((size,) * 3), 7) * 7
                     for _ in range(3)])
    warped = grid + disp
    moving_base = map_coordinates(fixed, warped, order=1, mode="nearest")
    moving_seg = map_coordinates(seg, warped, order=0, mode="nearest").astype(np.int16)
    fg = moving_seg > 0
    lo = moving_base[fg].min() if fg.any() else 0.0
    hi = moving_base[fg].max() if fg.any() else 1.0
    moving = np.zeros_like(moving_base)
    moving[fg] = (hi - moving_base[fg]) + lo            # contrast inversion in FG

    fixed_mask = binary_dilation(seg > 0, iterations=3).astype(np.uint8)
    moving_mask = binary_dilation(fg, iterations=3).astype(np.uint8)

    aff = np.eye(4)
    for name, arr in [("fixed", fixed), ("moving", moving.astype(np.float32)),
                      ("fixed_mask", fixed_mask), ("moving_mask", moving_mask),
                      ("fixed_seg", seg), ("moving_seg", moving_seg)]:
        nib.save(nib.Nifti1Image(arr, aff), os.path.join(outdir, name + ".nii.gz"))
    print("wrote demo volumes to", outdir, "(size %d^3)" % size)

make_demo_data()

## 3. Register with `anatomix-register.py`

T4-safe settings: the lightweight `anatomix` U-Net backbone, a single 128-voxel window (`sw_batch=1`), and a short 3-level pyramid. The default loss is `masked_cc` because we pass masks. For the full SOTA AbdomenMRCT configuration (`anatomix-dev-vit`, 8x4x2x1 / 100 iters), see the registration `README.md`.

In [ ]:
!python anatomix/registration/anatomix-register.py \
    --fixed demo/fixed.nii.gz --moving demo/moving.nii.gz \
    --fixed-mask demo/fixed_mask.nii.gz --moving-mask demo/moving_mask.nii.gz \
    --fixed-seg demo/fixed_seg.nii.gz --moving-seg demo/moving_seg.nii.gz \
    --backbone anatomix \
    --sliding-window-params 128,1,0.0,gaussian,0.25 \
    --transform deformable --step-size 1.0 \
    --shrink-factors 4x2x1 --iterations 50x50x50 --cc-kernel-widths 7x5x3 \
    --output-transformation-convention pytorch \
    --output-dir demo_out --exp-name demo

## 4. Inspect the results

`anatomix-register.py` wrote the moved image, the warped segmentation, the transform (`demo-warp-moving.pt`), and a `metrics.csv` with Dice and fold count. The Dice should rise well above the unregistered overlap.

In [ ]:
import glob
import csv
import nibabel as nib
import matplotlib.pyplot as plt

def load(p):
    return nib.load(p).get_fdata()

fixed = load("demo/fixed.nii.gz")
moving = load("demo/moving.nii.gz")
moved = load(sorted(glob.glob("demo_out/demo-moved-moving.nii.gz"))[0])
z = fixed.shape[2] // 2
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, (im, t) in zip(ax, [(fixed, "fixed (CT-like)"),
                            (moving, "moving (MR-like)"),
                            (moved, "moving warped to fixed")]):
    a.imshow(im[:, :, z].T, cmap="gray", origin="lower")
    a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

for row in csv.DictReader(open("demo_out/demo-metrics.csv")):
    print("Dice = %s   folds = %s" % (row["dice"], row["num_folds"]))

In [ ]:
# The saved transform is a normalized fixed->moving sampling grid; apply
# it to any volume on the moving grid with a single grid_sample call.
import torch, torch.nn.functional as F, nibabel as nib
grid = torch.load('demo_out/demo-warp-moving.pt')
moving = torch.tensor(nib.load('demo/moving.nii.gz').get_fdata())[None, None].float()
moved = F.grid_sample(moving, grid, mode='bilinear', align_corners=True)
print('applied transform ->', tuple(moved.shape))

### Next steps
- Swap `--backbone anatomix-dev-vit` for the strongest features (requires a 128-voxel window).
- Chain stages, e.g. `--initialization center-of-mass --transform affine,deformable`.
- Batch many pairs with `--fixed-dir/--moving-dir` or `--registration-pairs-csv`.
- See the registration `README.md` for the full CLI and the exact Learn2Reg-AbdomenMRCT reproduction command.